-- The hybrid approach is built here. First, we simulate the GRHyMoLAP on the 8 catchments abd then use the results as inout to train our LSTM. This code only shows the case of the hybrid approach incorporating temperature variables.

-- THe simulated Q and perc can be removed from the LSTM inputs, and in that case, the approaches become the standalone LSTM, as described in the paper.

# IMPORT LIBRARIES

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from matplotlib.dates import DateFormatter
from scipy.optimize import minimize # USE IN THE MODEL CALIBRATION

from google.colab import files
import zipfile
import os

In [ ]:
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


##CAMELS-DATA from my Google Drive.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd

# ==============================
# Paths to ZIP files
# ==============================
hydro_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/05_hydrometeorology.zip"
streamflow_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/03_streamflow.zip"

# Data extraction directories
hydro_dir = "/content/05_hydro"
streamflow_dir = "/content/03_streamflow"

# ==============================
# Function to extract ZIP files
# ==============================
def extract_zip(zip_path, extract_to):
    if not os.path.exists(extract_to):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✅ ZIP extracted in {extract_to}")
    else:
        print(f"✅ Directory {extract_to} already exists")

def find_csv(base_dir, csv_name):
    # Recursive search for the CSV file
    for root, dirs, files in os.walk(base_dir):
        if csv_name in files:
            return os.path.join(root, csv_name)
    raise FileNotFoundError(f"{csv_name} not found in {base_dir}")

# ==============================
# Extraction
# ==============================
extract_zip(hydro_zip, hydro_dir)
extract_zip(streamflow_zip, streamflow_dir)

# ==============================
# Load the 222 basins data
# ==============================
file_path = '/content/drive/MyDrive/Colab Notebooks/Dimension/id_name_metadata.csv'
basin222 = pd.read_csv(file_path)
station_ids_v1 = basin222['station_id'].astype(str).str.strip().unique()
print(f"✅ {len(station_ids_v1)} official basins loaded")

# ==============================
# 1️⃣ Precipitation (SILO)
# ==============================
precip_file = find_csv(hydro_dir, "precipitation_SILO.csv")
precip = pd.read_csv(precip_file, index_col=0, parse_dates=True)
precip.columns = precip.columns.str.strip()
precip.replace(-99.99, np.nan, inplace=True)
print("✅ SILO precipitation:", precip.shape)

# ==============================
# 2️⃣ Evapotranspiration (ET SILO)
# ==============================
et_file = find_csv(hydro_dir, "et_morton_actual_SILO.csv")
et = pd.read_csv(et_file, index_col=0, parse_dates=True)
et.columns = et.columns.str.strip()
et.replace(-99.99, np.nan, inplace=True)
print("✅ SILO ET:", et.shape)

# ==============================
# 3️⃣ Streamflow
# ==============================
streamflow_file = find_csv(streamflow_dir, "streamflow_mmd.csv")
Q = pd.read_csv(streamflow_file, index_col=0, parse_dates=True)
Q.columns = Q.columns.str.strip()
Q.replace(-99.99, np.nan, inplace=True)
print("✅ Streamflow:", Q.shape)

# ==============================
# 4️⃣ Maximum temperature (tmax)
# ==============================
tmax_file = find_csv(hydro_dir, "tmax_SILO.csv")
tmax = pd.read_csv(tmax_file, index_col=0, parse_dates=True)
tmax.columns = tmax.columns.str.strip()
tmax.replace(-99.99, np.nan, inplace=True)
print("✅ SILO tmax:", tmax.shape)

# ==============================
# 5️⃣ Minimum temperature (tmin)
# ==============================
tmin_file = find_csv(hydro_dir, "tmin_SILO.csv")
tmin = pd.read_csv(tmin_file, index_col=0, parse_dates=True)
tmin.columns = tmin.columns.str.strip()
tmin.replace(-99.99, np.nan, inplace=True)
print("✅ SILO tmin:", tmin.shape)

# ==============================
# 6️⃣ Vapor pressure deficit (vp_deficit)
# ==============================
vp_def_file = find_csv(hydro_dir, "vp_deficit_SILO.csv")
vp_deficit = pd.read_csv(vp_def_file, index_col=0, parse_dates=True)
vp_deficit.columns = vp_deficit.columns.str.strip()
vp_deficit.replace(-99.99, np.nan, inplace=True)
print("✅ SILO vp_deficit:", vp_deficit.shape)

# ==============================
# 7️⃣ Radiation
# ==============================
radiation_file = find_csv(hydro_dir, "radiation_SILO.csv")
radiation = pd.read_csv(radiation_file, index_col=0, parse_dates=True)
radiation.columns = radiation.columns.str.strip()
radiation.replace(-99.99, np.nan, inplace=True)
print("✅ SILO radiation:", radiation.shape)

# ==============================
# 8️⃣ Identify common stations (all variables)
# ==============================
stations_precip = set(precip.columns)
stations_et = set(et.columns)
stations_Q = set(Q.columns)
stations_tmax = set(tmax.columns)
stations_tmin = set(tmin.columns)
stations_vpdef = set(vp_deficit.columns)
stations_rad = set(radiation.columns)

common_stations = [
    s for s in station_ids_v1
    if s in stations_precip
    and s in stations_et
    and s in stations_Q
    and s in stations_tmax
    and s in stations_tmin
    and s in stations_vpdef
    and s in stations_rad
]

print(f"✅ Official common stations (all vars): {len(common_stations)}")

# ==============================
# 9️⃣ Subset common stations
# ==============================
precip = precip[common_stations]
et = et[common_stations]
Q = Q[common_stations]
tmax = tmax[common_stations]
tmin = tmin[common_stations]
vp_deficit = vp_deficit[common_stations]
radiation = radiation[common_stations]

# ==============================
# 🔟 Final verification
# ==============================
print("Precipitation:", precip.shape)
print("ET:", et.shape)
print("Streamflow:", Q.shape)
print("tmax:", tmax.shape)
print("tmin:", tmin.shape)
print("vp_deficit:", vp_deficit.shape)
print("radiation:", radiation.shape)
print("Stations (first 10):", common_stations[:10], "...")

In [ ]:
# Verification of the periods
print("Precipitation :", precip.index.min(), "→", precip.index.max())
print("ET            :", et.index.min(), "→", et.index.max())
print("Streamflow    :", Q.index.min(), "→", Q.index.max())


In [ ]:
# ==============================
# 0️⃣ Reduce all series to the period 1 January 1980 → 31 December 2014
# ==============================
start_date = "1980-01-01"
end_date   = "2014-12-31"

precip = precip.loc[start_date:end_date]
et     = et.loc[start_date:end_date]
Q      = Q.loc[start_date:end_date]

# Verification
print("Common period verification:")
print("Precipitation :", precip.index.min(), "→", precip.index.max())
print("ET            :", et.index.min(), "→", et.index.max())
print("Streamflow    :", Q.index.min(), "→", Q.index.max())

#GRHyMoLAP

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from numba import njit

# ============================================
# General parameters
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1

#stations = common_stations[:40]
stations = ['616002', '616013', 'A2390523', '408200', 'A0030501', 'A2390531', '614044',
 'A2390519']
results = {}

# ============================================
# Numba-accelerated functions
# ============================================
@njit
def Percolation_nb(Pn, En, X1):
    n = len(Pn)
    S = np.zeros(n)
    Perc = np.zeros(n)

    S[0] = X1 / 2
    ratio = (4.0 / 9.0) * (S[0] / X1)
    Perc[0] = S[0] * (1 - (1 + ratio**4) ** (-0.25))
    S[0] -= Perc[0]

    for t in range(1, n):
        temp = (S[t-1] / X1) ** 2

        frac = Pn[t] / X1
        Ps = X1 * (1 - temp) * np.tanh(frac) / (1 + (S[t-1] / X1) * np.tanh(frac))

        frac = En[t] / X1
        Es = S[t-1] * (2 - S[t-1] / X1) * np.tanh(frac) / (1 + (1 - S[t-1] / X1) * np.tanh(frac))

        S[t] = S[t-1] + Ps - Es

        ratio = (4.0 / 9.0) * (S[t] / X1)
        Perc[t] = S[t] * (1 - (1 + ratio**4) ** (-0.25))
        S[t] -= Perc[t]

    return Perc

@njit
def GRHyMoLAP_Model_nb(MU, LAMBDA, X1, gamma, Q0, Pn, En):
    N = len(Pn)
    Qsim = np.zeros(N)
    Qsim[0] = Q0

    Perc = Percolation_nb(Pn, En, X1)

    for t in range(N - 1):
        Qsim[t + 1] = max(
            0.0,
            Qsim[t] - (MU / LAMBDA) * Qsim[t] ** (2 * MU - 1) + gamma * Perc[t + 1] * Pn[t + 1]
        )

    return Qsim, Perc

# ============================================
# Vectorized NSE
# ============================================
def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs_valid = obs[mask]
    sim_valid = sim[mask]
    if len(obs_valid) == 0 or np.var(obs_valid) == 0:
        return np.nan
    return 1 - np.sum((sim_valid - obs_valid)**2) / np.sum((obs_valid - np.mean(obs_valid))**2)

def NNSE(nse):
    if np.isnan(nse):
        return np.nan
    return 1.0 / (2.0 - nse)

# ============================================
# Main loop over each basin
# ============================================
i = 0
for station_id in stations:
    i += 1
    print(f"\n=== Station {station_id} ===, Number = {i}")

    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    PET = et[station_id].loc[start_date:end_date].to_numpy(float)
    tmin1 = tmin[station_id].loc[start_date:end_date].to_numpy(float)
    tmax1 = tmax[station_id].loc[start_date:end_date].to_numpy(float)

    Pn = np.maximum(0, P - PET)
    En = np.maximum(0, PET - P)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        print("⚠️ Station skipped (no valid data).")
        continue

    missing_count = np.sum(np.isnan(Q_obs))
    missing_ratio = missing_count / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️ Too many missing values ({missing_ratio*100:.1f}%)")
        continue

    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0]

    # Objective for optimizer
    def objective(params, Q0, Pn_train, En_train, Q_obs_train):
        MU, LAMBDA, X1, gamma = params
        Qsim, _ = GRHyMoLAP_Model_nb(MU, LAMBDA, X1, gamma, Q0, Pn_train, En_train)
        nse = NSE(Q_obs_train, Qsim)
        return 1 - nse if np.isfinite(nse) else 1e9

    # Multi-start optimization
    initial_guesses = [
        [1.0, 8, 150, 0.1],
        [0.6, 2, 120, 1.0],
        [1.4, 15, 200, 0.5],
        [0.8, 4, 500, 0.2],
        [1.2, 10, 1000, 0.3],
        [0.7, 6, 350, 0.4],
        [1.1, 18, 20, 0.2],
        [0.9, 8, 2000, 0.6],
    ]

    best_res = None
    best_val = float("inf")

    for guess in initial_guesses:
        res = minimize(
            objective,
            guess,
            args=(Q0, Pn[:b1], En[:b1], Q_obs[:b1]),
            method="Nelder-Mead",
            #bounds=bounds,
            options={"maxiter": 2500, "disp": False}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    MU, LAMBDA, X1, GAMMA = best_res.x

    Qsim, Perc = GRHyMoLAP_Model_nb(MU, LAMBDA, X1, GAMMA, Q0, Pn, En)

    NSE_cal = NSE(Q_obs[:b1], Qsim[:b1])
    NSE_val = NSE(Q_obs[b1:], Qsim[b1:])
    NNSE_cal = NNSE(NSE_cal)
    NNSE_val = NNSE(NSE_val)

    print(f"✅ Calibration NSE: {NSE_cal:.3f}, Validation NSE: {NSE_val:.3f}")
    print(f"   Calibration NNSE: {NNSE_cal:.3f}, Validation NNSE: {NNSE_val:.3f}")
    print(f"   Params: mu={MU:.3f}, lambda={LAMBDA:.3f}, X1={X1:.3f}, gamma={GAMMA:.3f}")

    data_train = pd.DataFrame({
        "Qsim": Qsim[:b1],
        "perc": Perc[:b1],
        "prec": P[:b1],
        "pet": PET[:b1],
        "tmin": tmin1[:b1],
        "tmax": tmax1[:b1],
        "target": Q_obs[:b1]
    })
    data_test = pd.DataFrame({
        "Qsim": Qsim[b1:],
        "perc": Perc[b1:],
        "prec": P[b1:],
        "pet": PET[b1:],
        "tmin": tmin1[b1:],
        "tmax": tmax1[b1:],
        "target": Q_obs[b1:]
    })

    results[station_id] = {
        "params": [MU, LAMBDA, X1, GAMMA],
        "NSE_cal": NSE_cal,
        "NSE_val": NSE_val,
        "NNSE_cal": NNSE_cal,
        "NNSE_val": NNSE_val,
        "data_train": data_train,
        "data_test": data_test,
        "missing_ratio": missing_ratio,
        "missing_count": missing_count
    }

print(f"\n✅ Simulation completed for {len(results)} valid basins.")

NSE result summary

In [ ]:
# =============================================================
# 📌 EXTRACTION OF NSE & NNSE — CALIBRATION
# =============================================================
nse_cal = [res['NSE_cal'] for res in results.values()]
nnse_cal = [res['NNSE_cal'] for res in results.values()]

print("\n================= CALIBRATION =================\n")

if nse_cal:
    print(f"NSE Calibration -> Median: {np.percentile(nse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_cal, 95):.4f}")
    print("MEAN NSE_CAL:", np.mean(nse_cal))
    print("MIN NSE_CAL:", np.min(nse_cal))
    print("MAX NSE_CAL:", np.max(nse_cal))
else:
    print("No NSE available for the calibration.")

print("\n--- NNSE Calibration ---")
if nnse_cal:
    print(f"NNSE Calibration -> Median: {np.percentile(nnse_cal, 50):.3f}, "
          f"5th percentile: {np.percentile(nnse_cal, 5):.4f}, "
          f"95th percentile: {np.percentile(nnse_cal, 95):.4f}")
    print("MEAN NNSE_CAL:", np.mean(nnse_cal))
    print("MIN NNSE_CAL:", np.min(nnse_cal))
    print("MAX NNSE_CAL:", np.max(nnse_cal))
else:
    print("No NNSE avalaible for the calibration.")


# =============================================================
# 📌 EXTRACTION OF NSE & NNSE — VALIDATION
# =============================================================
print("\n\n================= VALIDATION =================\n")

nse_val = [res['NSE_val'] for res in results.values() if not np.isnan(res['NSE_val'])]
nnse_val = [res['NNSE_val'] for res in results.values() if not np.isnan(res['NNSE_val'])]

if nse_val:
    print(f"NSE Validation -> Median: {np.percentile(nse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nse_val, 95):.4f}")
    print("MEAN NSE_VAL:", np.mean(nse_val))
    print("MIN NSE_VAL:", np.min(nse_val))
    print("MAX NSE_VAL:", np.max(nse_val))
else:
    print("No valid station for NSE in validation.")

print("\n--- NNSE Validation ---")
if nnse_val:
    print(f"NNSE Validation -> Median: {np.percentile(nnse_val, 50):.3f}, "
          f"5th percentile: {np.percentile(nnse_val, 5):.4f}, "
          f"95th percentile: {np.percentile(nnse_val, 95):.4f}")
    print("MEAN NNSE_VAL:", np.mean(nnse_val))
    print("MIN NNSE_VAL:", np.min(nnse_val))
    print("MAX NNSE_VAL:", np.max(nnse_val))
else:
    print("No valid station for NNSE in validation.")

LSTM

In [ ]:
!pip install sympy==1.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 38.7 MB/s eta 0:00:00
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cpu requires sympy>=1.13.3, but you have sympy 1.12 which is incompatible.


In [ ]:
import sympy
print(sympy.__version__)

1.12


In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.8 MB/s eta 0:00:00


In [ ]:
#Libraries
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import optuna
import random
import copy

In [ ]:
# ==========================================================
# 🔁 Reproducibility
# ==========================================================
SEED = 142
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================================
# 1️⃣ NSE
# ==========================================================
def nash_sutcliffe_efficiency(obs, sim):
    obs = np.asarray(obs).flatten()
    sim = np.asarray(sim).flatten()
    mask = ~np.isnan(obs)
    obs = obs[mask]
    sim = sim[mask]
    denom = np.sum((obs - np.mean(obs)) ** 2)
    if denom == 0:
        return -np.inf
    return 1 - np.sum((obs - sim) ** 2) / denom

# ==========================================================
# 2️⃣ Data preparation (lag=5)
# ==========================================================
def prepare_data(data_train, data_test):
    def add_lag(df):
        df = df.copy()
        cols = ['prec','pet', 'tmin', 'tmax', 'Qsim', 'perc']
        for c in cols:
            df[c+"_lag1"] = df[c].shift(1)
            df[c+"_lag2"] = df[c].shift(2)
            df[c+"_lag3"] = df[c].shift(3)
            df[c+"_lag4"] = df[c].shift(4)
            df[c+"_lag5"] = df[c].shift(5)
        return df

    # 🔥 remove the first 5 rows
    train_df = add_lag(data_train).iloc[5:].reset_index(drop=True)
    val_df   = add_lag(data_test).iloc[5:].reset_index(drop=True)
    return train_df, val_df

# ==========================================================
# 3️⃣ Scaling
# ==========================================================
def scale_data(train_df, val_df, y_col="target"):
    X_cols = [
        'prec','pet', 'tmin', 'tmax', 'Qsim', 'perc',
        'prec_lag1','pet_lag1', 'tmin_lag1', 'tmax_lag1', 'Qsim_lag1', 'perc_lag1',
        'prec_lag2','pet_lag2', 'tmin_lag2', 'tmax_lag2', 'Qsim_lag2', 'perc_lag2',
        'prec_lag3','pet_lag3', 'tmin_lag3', 'tmax_lag3', 'Qsim_lag3', 'perc_lag3',
        'prec_lag4','pet_lag4', 'tmin_lag4', 'tmax_lag4', 'Qsim_lag4', 'perc_lag4',
        'prec_lag5','pet_lag5', 'tmin_lag5', 'tmax_lag5', 'Qsim_lag5', 'perc_lag5'
    ]

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    train_df[X_cols] = scaler_X.fit_transform(train_df[X_cols])
    val_df[X_cols]   = scaler_X.transform(val_df[X_cols])

    mask_train = ~train_df[y_col].isna()
    mask_val   = ~val_df[y_col].isna()

    train_df.loc[mask_train, y_col] = scaler_y.fit_transform(train_df.loc[mask_train, [y_col]])
    val_df.loc[mask_val, y_col]     = scaler_y.transform(val_df.loc[mask_val, [y_col]])

    return train_df, val_df, scaler_X, scaler_y, X_cols

# ==========================================================
# 4️⃣ DataLoader
# ==========================================================
def get_loader(df, X_cols, y_col="target", batch_size=32, shuffle=False):
    mask = ~df[y_col].isna()

    X = df.loc[mask, X_cols].values
    y = df.loc[mask, y_col].values

    X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    return DataLoader(TensorDataset(X.to(device), y.to(device)),
                      batch_size=batch_size, shuffle=shuffle)

# ==========================================================
# 5️⃣ LSTM
# ==========================================================
class SimpleLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

# ==========================================================
# 6️⃣ Training
# ==========================================================
def train_model(train_loader, val_loader, input_dim, scaler_y,
                epochs=200, hidden_dim=64, lr=1e-3):

    model = SimpleLSTM(input_dim, hidden_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_nse = -np.inf
    best_state = None

    for _ in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for Xv, yv in val_loader:
                preds.append(model(Xv))
                trues.append(yv)

        if len(preds) == 0:
            continue

        preds = torch.cat(preds).cpu().numpy()
        trues = torch.cat(trues).cpu().numpy()

        preds_phys = scaler_y.inverse_transform(preds)
        trues_phys = scaler_y.inverse_transform(trues)

        nse = nash_sutcliffe_efficiency(trues_phys, preds_phys)

        if nse > best_nse:
            best_nse = nse
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, best_nse

# ==========================================================
# 7️⃣ Basin loop
# ==========================================================
basins = list(results.keys())
print("📊 Number of basins:", len(basins))

all_cal = []
all_val = []

for st in basins:
    print(f"\n🔹 Basin {st}")

    data_train = results[st]["data_train"].copy()
    data_test  = results[st]["data_test"].copy()

    train_df, val_df = prepare_data(data_train, data_test)
    train_df, val_df, scaler_X, scaler_y, X_cols = scale_data(train_df, val_df)

    best_store = {}

    def objective(trial):
        hidden = trial.suggest_categorical("hidden", [8, 16, 32, 64, 128, 256])
        lr = trial.suggest_categorical("lr", [1e-2, 1e-3])
        epochs = trial.suggest_int("epochs", 100, 500, step=100)
        batch = trial.suggest_categorical("batch", [8, 16, 32, 64, 128, 256])

        train_loader = get_loader(train_df, X_cols, batch_size=batch, shuffle=True)
        val_loader   = get_loader(val_df, X_cols, batch_size=batch, shuffle=False)

        model, best_nse = train_model(
            train_loader, val_loader,
            input_dim=len(X_cols),
            scaler_y=scaler_y,
            epochs=epochs,
            hidden_dim=hidden,
            lr=lr
        )

        best_store[trial.number] = {
            "state": copy.deepcopy(model.state_dict()),
            "best_nse": best_nse,
            "params": dict(hidden=hidden, lr=lr, epochs=epochs, batch=batch)
        }
        return best_nse

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)

    best = best_store[study.best_trial.number]
    model_final = SimpleLSTM(len(X_cols), best["params"]["hidden"]).to(device)
    model_final.load_state_dict(best["state"])
    model_final.eval()

    mask_train = ~train_df["target"].isna()
    mask_val   = ~val_df["target"].isna()

    with torch.no_grad():
        y_train_pred = np.full(len(train_df), np.nan)
        y_val_pred   = np.full(len(val_df), np.nan)

        if mask_train.any():
            Xtr = torch.tensor(train_df.loc[mask_train, X_cols].values,
                               dtype=torch.float32).unsqueeze(1).to(device)
            y_train_pred[mask_train] = scaler_y.inverse_transform(
                model_final(Xtr).cpu().numpy()).ravel()

        if mask_val.any():
            Xv = torch.tensor(val_df.loc[mask_val, X_cols].values,
                              dtype=torch.float32).unsqueeze(1).to(device)
            y_val_pred[mask_val] = scaler_y.inverse_transform(
                model_final(Xv).cpu().numpy()).ravel()

    dates_train = train_df["date"].values if "date" in train_df.columns else np.arange(len(train_df))
    dates_val   = val_df["date"].values if "date" in val_df.columns else np.arange(len(val_df))

    all_cal.append(pd.DataFrame({"Date": dates_train, "Basin": st, "Pred": y_train_pred}))
    all_val.append(pd.DataFrame({"Date": dates_val, "Basin": st, "Pred": y_val_pred}))

    print(f"✅ NSE = {best['best_nse']:.4f}")

# ==========================================================
# 🔄 Pivot
# ==========================================================
df_cal = pd.concat(all_cal).pivot(index="Date", columns="Basin", values="Pred").reset_index()
df_val = pd.concat(all_val).pivot(index="Date", columns="Basin", values="Pred").reset_index()

# ==========================================================
# 💾 Export
# ==========================================================
file_name = "Hybrid_LSTM_lag5.xlsx"
with pd.ExcelWriter(file_name) as writer:
    df_cal.to_excel(writer, sheet_name="Calibration", index=False)
    df_val.to_excel(writer, sheet_name="Validation", index=False)

print(f"\n📁 Excel file saved: {file_name}")

from google.colab import files
#files.download(file_name)